# SpanNER + DinGenPipe (Two-Stage Reflective Multi-Agent LinkNER)

This notebook implements and audits the **Two-Stage Reflective Multi-Agent LinkNER Architecture** on Lifestyle Factors (LSF) biomedical abstracts.

---

### 🏗️ Unified Architecture Workflow

```
                                      ┌────────────────────────────────────────────────────────┐
                                      │               Raw Biomedical Abstract                  │
                                      └──────────────────────────┬─────────────────────────────┘
                                                                 │
                              ┌──────────────────────────────────┴──────────────────────────────────┐
                              ▼                                                                     ▼
               ┌──────────────────────────────┐                                      ┌──────────────────────────────┐
               │   Stage 1: SpanNER Model     │                                      │   Stage 2: LOF Novelty       │
               │  Evaluates candidate spans   │                                      │  Computes vector embeddings  │
               │  Uncertainty (U_SpanNER)     │                                      │  & LOF distance to exemplars │
               └──────────────┬───────────────┘                                      └──────────────┬───────────────┘
                              │                                                                     │
                              └──────────────────────────────────┬──────────────────────────────────┘
                                                                 ▼
                                              ┌─────────────────────────────────────┐
                                              │    Stage 3: Composite Gating        │
                                              │  Unreliability = 0.2*Nov + 0.8*Unc  │
                                              └──────────┬────────────────┬─────────┘
                                                         │                │
                                    Unreliability < 0.40 │                │ Unreliability >= 0.40
                                                         ▼                ▼
                                            ┌──────────────────┐  ┌──────────────────────────────────────────────┐
                                            │   LOCAL SPANNER  │  │  Stage 4a: Dinasor (Reflection Agent)        │
                                            │  Zero API Cost   │  │  Formulates ~5 targeted disambiguation hints │
                                            │  Local Decision  │  └──────────────────────┬───────────────────────┘
                                            └──────────────────┘                         │
                                                                                         ▼
                                                                  ┌──────────────────────────────────────────────┐
                                                                  │  Stage 4b: DinGenerator (Resolution Agent)   │
                                                                  │  Resolves canonical categories & boundaries  │
                                                                  └──────────────────────┬───────────────────────┘
                                                                                         │
                                                                                         ▼
                                                                  ┌──────────────────────────────────────────────┐
                                                                  │  Stage 4c: Margin Safeguard & Arbitration    │
                                                                  │  Vetoes false drops on decisive SpanNER spans│
                                                                  └──────────────────────┬───────────────────────┘
                                                                                         │
                                                                                         ▼
                                                                  ┌──────────────────────────────────────────────┐
                                                                  │  Stage 5: 8-Outcome Audit & 4-Tier Benchmark │
                                                                  └──────────────────────────────────────────────┘
```

---

In [ ]:
%load_ext autoreload
%autoreload 2

# ==============================================================================
# [CELL 1]: Setup, Environment, .env Loading & Path Resolution
# ==============================================================================
import os
import sys
import glob
import json
import re
import math
import time
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

warnings.filterwarnings("ignore")
try:
    import urllib3
    warnings.filterwarnings("ignore", category=urllib3.exceptions.NotOpenSSLWarning)
except Exception:
    pass

from dotenv import load_dotenv, find_dotenv
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, RobertaForTokenClassification
from src.data.cached_loader import load_cached_spanner_candidates, get_cache_split_key

# 1. Path Bootstrapping for Unified_Test Workspace
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
WORKSPACE_ROOT = PROJECT_ROOT.parent
LSF_ROOT = PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# 2. Load API Keys and Master Parameters from Unified_Test/.env
DOTENV_PATH = PROJECT_ROOT / ".env"
if DOTENV_PATH.exists():
    load_dotenv(DOTENV_PATH, override=True)
    print(f"✅ Loaded environment & API keys from: {DOTENV_PATH}")
else:
    load_dotenv(find_dotenv(), override=True)
    print("⚠️ Loaded environment from fallback .env file.")

# 3. Direct Workspace Imports (No redundant try/except)
from src.architectures.dinasor_dingen.dinasor_agent import Dinasor
from src.architectures.dinasor_dingen.dingen_agent import DinGenerator
from src.architectures.hybrid_linkner.spanner_model import SpanNERModel
from src.architectures.hybrid_linkner.novelty_detector import build_lof_novelty_detector, LOFNoveltyDetector
from src.data.dataset_converter import parse_ann_file
from src.common.label_mapping import detect_schema_from_path, canonicalize_label, to_new_label, to_old_label
from src.common.audit_logger import (
    classify_audit_outcome,
    get_audit_status,
    compute_audit_breakdown,
    format_audit_breakdown_table,
    format_audit_summary_box,
    AUDIT_CATEGORIES,
    compute_canonical_audit_breakdown,
    format_canonical_audit_table,
    format_4_formulations_summary_box,
    create_run_directory,
    log_abstract_audit,
    log_run_summary,
)
from src.common.evaluation_framework import (
    evaluate_doc_4_formulations,
    aggregate_4_formulations,
    format_4_formulations_summary_box,
)
from src.architectures.dinasor_dingen.pipeline import SpanNER_DinGenPipeline, build_anchored_abstract

print("✓ Setup and environment initialization complete.")


In [ ]:
# ==============================================================================
# [CELL 2]: Master Experiment & Pipeline Configurations (from .env)
# ==============================================================================
SEED = int(os.getenv("RANDOM_SEED", "42"))
torch.manual_seed(SEED)
np.random.seed(SEED)

# 1. Model & LLM Provider Configuration from .env
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "9router")
MODEL_NAME = os.getenv("LLM_MODEL", os.getenv("OPENROUTER_MODEL", "gemini-cli"))
BACKEND = os.getenv("LLM_BACKEND", "api")  # Options: 'api', 'lmstudio', 'ollama', 'dummy'
LLM_TEMPERATURE = float(os.getenv("LLM_TEMPERATURE", "0.0"))
LLM_MAX_TOKENS = int(os.getenv("LLM_MAX_TOKENS", "4096"))
PROMPT_VERSION = os.getenv("PROMPT_VERSION", "v3")  # Options: "v1" (ablation baseline), "v2" (enriched), "v3" (optimized hybrid)

OUTPUT_FOLDER = os.path.abspath(os.path.join(PROJECT_ROOT, "output/dinasor_dingen/sequential"))
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# 2. Dataset Paths: Priority to Balanced Validation Set (80 Abstracts)
candidate_data_dirs = [
    os.path.join(PROJECT_ROOT, "data/new/400Abstracts/no_disease/balanced/val"),
    os.path.join(PROJECT_ROOT, "data/splits/val"),
    os.path.join(PROJECT_ROOT, "data/new/200Abstracts/test"),
    os.path.join(PROJECT_ROOT, "data/raw/Relabaled_test_folder_noDisease"),
]
DATA_DIR = next((d for d in candidate_data_dirs if os.path.exists(d)), candidate_data_dirs[0])

# Auto-detect label schema from dataset path
try:
    USE_NEW_LABELS = detect_schema_from_path(DATA_DIR)
except Exception:
    USE_NEW_LABELS = True

# 3. Gating Parameters & Thresholds (Optimal Configuration from Ablation Study)
BATCH_SIZE = int(os.getenv("BATCH_SIZE", "5"))
METHOD = "margin"  # Optimal: Margin uncertainty
GATING_MODE = "prob_or"  # Optimal: Probabilistic OR gating
UNRELIABILITY_THRESHOLD = 0.35  # Optimal: Tau = 0.35
NOVELTY_THRESHOLD = 0.55
W_NOVELTY = 0.30  # Optimal: w_nov = 0.30
W_UNCERTAINTY = 0.70  # Optimal: w_unc = 0.70
ARBITRATION_MARGIN_THRESHOLD = 0.50  # Optimal: Margin Safeguard = 0.50
APPLY_NMS = os.getenv("APPLY_NMS", "False").strip().lower() in ("true", "1", "yes")  # Default False to preserve nested entities

# 4. Embedding Configuration for LOF Novelty Detector (.env Local Endpoint / Offline Fallback)
EMBEDDING_PROVIDER = os.getenv("EMBEDDING_PROVIDER", "9router")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "un/Qwen3-Embedding-8B")
EMBEDDING_BASE_URL = os.getenv("EMBEDDING_BASE_URL", "http://localhost:20128/v1")
QWEN_LOCAL_PATH = os.path.join(PROJECT_ROOT, "models/Qwen3-Embedding-8B")
if not os.path.exists(QWEN_LOCAL_PATH):
    QWEN_LOCAL_PATH = "/Users/hmd/Documents/Workspace/HuggingFaceDownloader/models/Qwen3-Embedding-8B"

os.environ["QWEN_EMBEDDING_PATH"] = QWEN_LOCAL_PATH
ENABLE_NOVELTY_DETECTOR = True

schema_name = "NEW (Relabeled)" if USE_NEW_LABELS else "LEGACY"
print("=" * 80)
print("  📋 EXPERIMENT CONFIGURATION (Loaded from .env)")
print("=" * 80)
print(f"  • Data Directory         : {DATA_DIR}")
print(f"  • Label Schema           : {schema_name} (USE_NEW_LABELS = {USE_NEW_LABELS})")
print(f"  • Active LLM Provider    : {LLM_PROVIDER.upper()}")
print(f"  • Active Model Name      : {MODEL_NAME} (Backend: {BACKEND})")
print(f"  • Prompt Version         : {PROMPT_VERSION.upper()}")
print(f"  • Temperature / MaxTok   : {LLM_TEMPERATURE} / {LLM_MAX_TOKENS}")
print(f"  • Uncertainty Method     : {METHOD.upper()}")
print(f"  • Gating Strategy        : {GATING_MODE.upper()} (Unreliability Thresh = {UNRELIABILITY_THRESHOLD})")
print(f"  • Weights (Nov / Unc)    : {W_NOVELTY} / {W_UNCERTAINTY}")
print(f"  • Margin Safeguard Veto  : {ARBITRATION_MARGIN_THRESHOLD}")
print(f"  • Embedding Model        : {EMBEDDING_MODEL} (Local Path: {QWEN_LOCAL_PATH})")
print(f"  • Output Logs Directory  : {OUTPUT_FOLDER}")
print(f"  • Apply NMS (Boundary Filter): {APPLY_NMS}")
print("=" * 80)

# Package Full Experiment Configuration for Audit Logging
EXPERIMENT_CONFIG = {
    'SEED': SEED,
    'DATA_DIR': DATA_DIR,
    'USE_NEW_LABELS': USE_NEW_LABELS,
    'LLM_PROVIDER': LLM_PROVIDER,
    'MODEL_NAME': MODEL_NAME,
    'METHOD': METHOD,
    'GATING_MODE': GATING_MODE,
    'UNRELIABILITY_THRESHOLD': UNRELIABILITY_THRESHOLD,
    'NOVELTY_THRESHOLD': NOVELTY_THRESHOLD,
    'W_NOVELTY': W_NOVELTY,
    'W_UNCERTAINTY': W_UNCERTAINTY,
    'ARBITRATION_MARGIN_THRESHOLD': ARBITRATION_MARGIN_THRESHOLD,
    'OUTPUT_FOLDER': OUTPUT_FOLDER,
    'APPLY_NMS': APPLY_NMS
}

# 5. Cached SpanNER Output Configuration (High-Speed & Exact Reproducibility)
USE_CACHED_SPANNER = os.getenv("USE_CACHED_SPANNER", "true").lower() in ("true", "1", "yes")
CACHED_SPANNER_DIR = os.path.join(PROJECT_ROOT, "data/cached/spanner_160_10_class")


In [ ]:
# ==============================================================================
# [CELL 3]: SpanNER Neural Model, LOF Novelty & Agent Initialization
# ==============================================================================
device = torch.device("mps") if torch.backends.mps.is_available() else (torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
print(f"🖥️ Computation Device: {device}")

# 1. Resolve Model Checkpoint Paths
SPANNER_ENCODER_PATH = os.path.join(PROJECT_ROOT, "models/NER_Model/trained_NER_model")
SPANNER_CHECKPOINT_PATH = os.path.join(PROJECT_ROOT, "models/SpanNER_LSF/best_spanner_160train.pt")

# Standard 11-class SpanNER mapping
ID2LABEL = {
    0: 'O',
    1: 'Personal_care_products_and_cosmetic_procedures',
    2: 'Substance_use',
    3: 'Environmental_exposures',
    4: 'Mental_health_practices',
    5: 'Non_physical_leisure_time_activities',
    6: 'Nutrition',
    7: 'Physical_activities',
    8: 'Sleep',
    9: 'Socioeconomic_factors'
}
NUM_CLASSES = len(ID2LABEL)

# Load SpanNER Model and Tokenizer
tokenizer = AutoTokenizer.from_pretrained(SPANNER_ENCODER_PATH)
spanner_model = SpanNERModel(encoder_path=SPANNER_ENCODER_PATH, num_classes=NUM_CLASSES, max_span_width=6)
spanner_model.load_state_dict(torch.load(SPANNER_CHECKPOINT_PATH, map_location=device))
spanner_model.to(device)
spanner_model.eval()
print(f"✅ SpanNER model loaded successfully from {SPANNER_CHECKPOINT_PATH}")

# 2. Build and Fit LOF Novelty Detector (Via Local Endpoint or Local Fallback)
if ENABLE_NOVELTY_DETECTOR:
    if EMBEDDING_PROVIDER in ("9router", "api", "gateway"):
        print(f"📡 Connecting to local Qwen embedding endpoint: {EMBEDDING_BASE_URL} (Model: '{EMBEDDING_MODEL}')...")
    else:
        print(f"🔄 Loading local Qwen weights from disk: {QWEN_LOCAL_PATH}...")
    novelty_detector = build_lof_novelty_detector(k=10, alpha=0.5)
    print(f"✅ LOF Novelty Detector active with {novelty_detector.embedder}!")
    print(f"   Fitted on {len(novelty_detector.matrix)} training exemplars from vector DB.")
else:
    print("ℹ️ Novelty detector safely disabled.")
    novelty_detector = None

# 3. Initialize Dinasor & DinGenerator Agents
dinasor_agent = Dinasor(model_name=MODEL_NAME, backend=BACKEND, prompt_version=PROMPT_VERSION)
dingenerator_agent = DinGenerator(model_name=MODEL_NAME, backend=BACKEND, prompt_version=PROMPT_VERSION)
print(f"✅ Dinasor and DinGenerator agents initialized (Backend: '{BACKEND}', Model: '{MODEL_NAME}', Prompt: '{PROMPT_VERSION}')")


In [ ]:
# ==============================================================================
# [CELL 4]: Load Dataset Abstracts & Ground Truth Annotations
# ==============================================================================
def normalize_text(text: str) -> str:
    return text.strip().lower()

def match_gt_label(span_text: str, start_char: int, end_char: int, gt_ents: list, pred_label: str = None) -> str:
    p_norm = normalize_text(span_text)
    # 1. Exact character boundaries first
    for g in gt_ents:
        if start_char == g.get("start_char", -1) and end_char == g.get("end_char", -1):
            return g.get("label", "O")
    # 2. Nested entity aware match with pred_label
    if pred_label:
        from src.common.label_mapping import canonicalize_label as c_lbl
        p_c = c_lbl(pred_label)
        for g in gt_ents:
            g_c = c_lbl(g.get("label", "O"))
            overlap = not (end_char <= g.get("start_char", 0) or start_char >= g.get("end_char", 0))
            if overlap and p_c == g_c:
                return g.get("label", "O")
    # 3. Position proximity overlap fallback
    for g in gt_ents:
        g_norm = normalize_text(g.get("text", ""))
        overlap = not (end_char <= g.get("start_char", 0) or start_char >= g.get("end_char", 0))
        if overlap and (p_norm in g_norm or g_norm in p_norm):
            return g.get("label", "O")
    return "O"

all_ann = sorted(glob.glob(os.path.join(DATA_DIR, "*.ann")))
selected_files = []

for ann_path in all_ann:
    gt_ents = parse_ann_file(ann_path)
    txt_path = ann_path.replace(".ann", ".txt")
    if os.path.exists(txt_path):
        selected_files.append((txt_path, ann_path, gt_ents))

print(f"✅ Successfully loaded {len(selected_files)} paired abstracts (.txt & .ann) from {DATA_DIR}")

In [ ]:
# ==============================================================================
# [CELL 5]: Instantiate SpanNER_DinGenPipeline Engine
# ==============================================================================
import importlib
import src.architectures.dinasor_dingen.pipeline as pipe_module
importlib.reload(pipe_module)
from src.architectures.dinasor_dingen.pipeline import SpanNER_DinGenPipeline, build_anchored_abstract

pipeline = SpanNER_DinGenPipeline(
    spanner_model=spanner_model,
    tokenizer=tokenizer,
    novelty_scorer=novelty_detector,
    dinasor_agent=dinasor_agent,
    dingenerator_agent=dingenerator_agent,
    device=device,
    id2label=ID2LABEL,
    use_new_labels=USE_NEW_LABELS,
    w_novelty=W_NOVELTY,
    w_uncertainty=W_UNCERTAINTY,
    unreliability_threshold=UNRELIABILITY_THRESHOLD,
    novelty_threshold=NOVELTY_THRESHOLD,
    margin_threshold=ARBITRATION_MARGIN_THRESHOLD,
    gating_mode=GATING_MODE,
    output_folder=OUTPUT_FOLDER,
    apply_nms=APPLY_NMS,
    enable_cache=True
)

print(f"✓ Modular SpanNER_DinGenPipeline engine initialized successfully. (LLM Cache: {len(pipeline.llm_cache)} active entries)")


In [ ]:
# ==============================================================================
# [CELL 6]: Single Abstract Interactive Demonstration & Visual Trace
# ==============================================================================
sample_txt_path, sample_ann_path, sample_gt_ents = selected_files[0]
sample_doc_id = os.path.basename(sample_txt_path).replace(".txt", "")

with open(sample_txt_path, "r", encoding="utf-8") as f:
    sample_abstract = f.read()

print("=" * 80)
print(f"📄 DEMO: Running SpanNER + DinGenPipe on Abstract: {sample_doc_id}")
print("=" * 80)

sample_cands = None
if USE_CACHED_SPANNER:
    split_key = get_cache_split_key(DATA_DIR)
    demo_cache = load_cached_spanner_candidates(split_key=split_key, cache_dir=CACHED_SPANNER_DIR, doc_ids=[sample_doc_id])
    sample_cands = demo_cache.get(sample_doc_id, [])

sample_preds = pipeline.predict_abstract(sample_abstract, uncertainty_method=METHOD, doc_id=sample_doc_id, precomputed_candidates=sample_cands)

# Display Visual Comparison DataFrame
rows = []
for p in sample_preds:
    span_text = p["span_text"]
    is_esc = p.get("escalated_to_llm", False)
    s_lbl = p.get("spanner_label", "O")
    l_lbl = p.get("llm_label", "O")
    f_lbl = p.get("label", "O")
    gt = match_gt_label(span_text, p["start_char"], p["end_char"], sample_gt_ents, pred_label=f_lbl)
    
    try:
        audit_res = classify_audit_outcome(s_lbl, l_lbl, f_lbl, gt)
        if isinstance(audit_res, (list, tuple)) and len(audit_res) >= 2:
            cat_name = audit_res[1]
        elif isinstance(audit_res, str):
            cat_name = audit_res.replace("_", " ").title()
        else:
            cat_name = "🚀 Category Correction" if f_lbl == gt else "❌ Decision Error"
    except Exception:
        cat_name = "🚀 Fixed" if f_lbl == gt else "❌ Error"
    
    rows.append({
        "Span": span_text,
        "SpanNER Tag": s_lbl,
        "Uncertainty": round(float(p.get("uncertainty", p.get("u_score", 0.0))), 4),
        "Novelty": round(float(p.get("novelty", 0.0)), 4),
        "Unreliability": round(float(p.get("unreliability", 0.0)), 4),
        "Escalated": "⚠️ YES" if is_esc else "⚡ Local",
        "DinGen Resolved": f_lbl,
        "Ground Truth": gt,
        "Audit Status": cat_name if is_esc else ("✅ Correct" if f_lbl.lower() == gt.lower() else "❌ Local Error")
    })

demo_df = pd.DataFrame(rows)
print(f"\nTotal Entities Extracted: {len(sample_preds)}")
display(demo_df) if 'display' in globals() else print(demo_df)


In [ ]:
# ==============================================================================
# [CELL 7]: Batch Inference & Escalation Audit across Dataset
# ==============================================================================
from src.architectures.dinasor_dingen.pipeline import build_anchored_abstract
from src.common.audit_logger import create_run_directory, log_abstract_audit, log_run_summary

# Initialize Run Directory for detailed logs
run_name = f"dingen_pipe_prompt_{PROMPT_VERSION.lower()}_{GATING_MODE.lower()}_nov_{W_NOVELTY:.2f}_unc_{W_UNCERTAINTY:.2f}_tau_{UNRELIABILITY_THRESHOLD:.2f}"
run_dir, abstracts_dir, run_id = create_run_directory(OUTPUT_FOLDER, run_name=run_name)

all_texts = []
all_doc_ids = []
all_doc_gts = []

for txt_path, ann_path, gt_ents in selected_files:
    with open(txt_path, "r", encoding="utf-8") as f:
        all_texts.append(f.read())
    all_doc_ids.append(os.path.basename(txt_path).replace(".txt", ""))
    all_doc_gts.append(gt_ents)

print("=" * 96)
print(f"  Auditing Generative LLM Escalations in Batches (SpanNER + DinGenPipe | Method: {METHOD.upper()}, Gating: {GATING_MODE.upper()}, Threshold = {UNRELIABILITY_THRESHOLD}, B = {BATCH_SIZE})")
print("=" * 96)
print(f"📁 Created dedicated run logging directory: {run_dir}")

t0 = time.time()
batch_cands = None
if USE_CACHED_SPANNER:
    print(f"🚀 Loading precomputed SpanNER candidates from cache: {CACHED_SPANNER_DIR}...")
    split_key = get_cache_split_key(DATA_DIR)
    batch_cands = load_cached_spanner_candidates(
        split_key=split_key,
        cache_dir=CACHED_SPANNER_DIR,
        doc_ids=all_doc_ids
    )
    total_cached = sum(len(d.get("candidates", [])) for d in batch_cands.values())
    print(f"⚡ [CACHE HIT] Loaded {total_cached} precomputed candidate spans across {len(batch_cands)} abstracts.")

all_doc_preds = pipeline.predict_batch(
    texts=all_texts,
    batch_size=BATCH_SIZE,
    uncertainty_method=METHOD,
    doc_ids=all_doc_ids,
    verbose=True,
    precomputed_candidates=batch_cands
)
total_elapsed = time.time() - t0

# 3. Generate Detailed Per-Abstract and Run Summary Logs
all_audit_records = []
all_abstract_metrics = []
escalated_audit_items = []

print(f"\n📝 Writing detailed step-by-step audit logs to {abstracts_dir}...")
for doc_id, text, gt_ents, preds in zip(all_doc_ids, all_texts, all_doc_gts, all_doc_preds):
    dinasor_hints = []
    dinasor_prompt = ""
    dinasor_raw = ""
    dingen_prompt = ""
    dingen_raw = ""
    dingen_res = {'labeled_entities': []}
    unc_spans = []
    conf_spans = []
    for p in preds:
        span_text = p["span_text"]
        is_esc = p.get("escalated_to_llm", False)
        s_lbl = p.get("spanner_label", "O")
        l_lbl = p.get("llm_label", "O")
        f_lbl = p.get("label", "O")
        gt = match_gt_label(span_text, p["start_char"], p["end_char"], gt_ents, pred_label=f_lbl)
        
        if is_esc:
            escalated_audit_items.append({
                'span': span_text,
                'spanner_label': s_lbl,
                'llm_label': l_lbl,
                'final_label': f_lbl,
                'ground_truth': gt,
                'ground_truth_label': gt,
                'u_score': p.get('uncertainty', p.get('u_score', 0.0)),
                'novelty': p.get('novelty', 0.0),
                'unreliability': p.get('unreliability', 0.0),
                'doc_id': doc_id
            })
            unc_spans.append(p)
            if p.get('dinasor_hints'):
                dinasor_hints = p['dinasor_hints']
            if p.get('dinasor_prompt'):
                dinasor_prompt = p['dinasor_prompt']
            if p.get('dinasor_raw_response'):
                dinasor_raw = p['dinasor_raw_response']
            if p.get('dingenerator_prompt'):
                dingen_prompt = p['dingenerator_prompt']
            if p.get('dingenerator_raw_response'):
                dingen_raw = p['dingenerator_raw_response']
            dingen_res['labeled_entities'].append({
                'entity': p['span_text'],
                'label': p.get('llm_label', p['label']),
                'rationale': p.get('rationale', 'Resolved by DinGenerator')
            })
        else:
            conf_spans.append(p)
            
    anchored_doc = build_anchored_abstract(text, unc_spans, conf_spans)
    
    log_path, metrics, records = log_abstract_audit(
        doc_id=doc_id,
        text=text,
        gt_ents=gt_ents,
        preds=preds,
        dinasor_hints=dinasor_hints,
        dingen_results=dingen_res,
        anchored_abstract=anchored_doc,
        config=EXPERIMENT_CONFIG,
        abstracts_dir=abstracts_dir,
        dinasor_prompt=dinasor_prompt,
        dingenerator_prompt=dingen_prompt,
        dinasor_raw_response=dinasor_raw,
        dingenerator_raw_response=dingen_raw
    )
    all_audit_records.extend(records)
    all_abstract_metrics.append(metrics)

summary_txt, summary_json = log_run_summary(
    all_audit_records=all_audit_records,
    all_abstract_metrics=all_abstract_metrics,
    config=EXPERIMENT_CONFIG,
    execution_time=total_elapsed,
    run_dir=run_dir
)

print(f"\n" + "=" * 88)
print(f"  ✅ AUDIT LOGGING COMPLETE!")
print(f"  • Run Directory : {run_dir}")
print(f"  • Abstract Logs : {len(all_doc_ids)} files in {abstracts_dir}")
print(f"  • Summary TXT   : {summary_txt}")
print(f"  • Summary JSON  : {summary_json}")
print("=" * 88)


In [ ]:
# ==============================================================================
# [CELL 7b]: Alternative Fast Test Phase - TypeSafe System One (Jev Mode 2) Async Parallel Inference
# High-throughput asynchronous evaluation (~12s runtime across all 80 abstracts)
# ==============================================================================
import asyncio
import concurrent.futures
import time
import json
from pathlib import Path
import httpx
from src.architectures.JeBert import JevGenerator
from src.common.label_mapping import canonicalize_label

USE_JEV_INFERENCE = True  # Toggle True to evaluate Jev System One; False to preserve baseline
JEV_MODE = "two_stage"    # Supported: "two_stage" (Pass 1: Rule Assignment -> Pass 2: Scoped Classification), "mode2", "mode3", "mode1"
USE_CACHE_IF_AVAILABLE = True  # Instant offline loading from bertized_ace_checkpoint_cache.json if present

if USE_JEV_INFERENCE:
    print("=" * 96)
    print(f"⚡ Running High-Throughput Async Inference with TypeSafe System One (Jev: {JEV_MODE}) on {len(all_doc_ids)} abstracts...")
    print("=" * 96)

    cache_file = Path("Jev/results/bertized_ace_checkpoint_cache.json")
    cache_data = {}
    if USE_CACHE_IF_AVAILABLE and cache_file.exists():
        try:
            with open(cache_file, "r", encoding="utf-8") as f:
                cache_data = json.load(f)
            print(f"📦 Loaded pre-computed Jev checkpoint cache ({len(cache_data)} abstracts available for instant evaluation).")
        except Exception as e:
            print(f"⚠️ Could not load cache: {e}")

    jev_gen = JevGenerator(mode=JEV_MODE)
    t0_jev = time.time()

    async def _run_jev_dingen_suite(concurrency: int = 8):
        sem = asyncio.Semaphore(concurrency)
        async with httpx.AsyncClient(timeout=30.0) as http_client:
            async def _process_abstract(doc_id, text):
                cands_data = batch_cands.get(doc_id, {}) if batch_cands else {}
                cands = cands_data.get("candidates", [])
                
                # Filter through unreliability threshold
                escalated_cands, confident_cands = [], []
                tau = float(getattr(pipeline, "unreliability_threshold", 0.40))
                for c in cands:
                    unrel = float(c.get("unreliability", c.get("u_score", 0.0)))
                    if unrel >= tau:
                        escalated_cands.append(c)
                    else:
                        confident_cands.append(c)
                
                doc_cache = cache_data.get(doc_id, {}).get(JEV_MODE, {})
                if doc_cache and all(c.get("span_id") in doc_cache for c in escalated_cands):
                    ans_map = {sid: {"id": sid, "final_label": data["choice"], "confidence": data.get("confidence", 1.0)} for sid, data in doc_cache.items()}
                else:
                    async with sem:
                        res = await jev_gen.run_async(abstract=text, candidate_spans=escalated_cands, http_client=http_client)
                    ans_map = {a.get("id"): a for a in res.get("llm_answer", [])}
                
                doc_preds = []
                for idx, c in enumerate(escalated_cands):
                    cid = c.get("span_id", c.get("id"))
                    ans = ans_map.get(cid) if cid is not None else None
                    if not ans and idx < len(ans_map):
                        ans = list(ans_map.values())[idx]
                    
                    s_lbl = canonicalize_label(c.get("spanner_label", c.get("predicted_label", "O")), use_new_labels=USE_NEW_LABELS)
                    raw_lbl = canonicalize_label(ans.get("final_label", s_lbl) if ans else s_lbl, use_new_labels=USE_NEW_LABELS)
                    
                    cand_margin = float(c.get("margin", c.get("prob", 0.0)))
                    cand_p_o = float(c.get("p_o", 0.0))
                    tau_margin = float(getattr(pipeline, "margin_threshold", 0.85))
                    if raw_lbl == "O" and cand_margin >= tau_margin and s_lbl.lower() != "o" and cand_p_o < 0.20:
                        final_lbl = s_lbl
                        src = "SpanNER Margin Veto"
                    else:
                        final_lbl = raw_lbl
                        src = "TypeSafe-Jev (mode2)"
                    
                    doc_preds.append({
                        "span_text": c.get("span_text", ""),
                        "start_char": c.get("start_char", 0),
                        "end_char": c.get("end_char", 0),
                        "label": final_lbl,
                        "spanner_label": s_lbl,
                        "llm_label": raw_lbl,
                        "uncertainty": float(c.get("uncertainty", c.get("u_score", 0.0))),
                        "novelty": float(c.get("novelty_score", c.get("novelty", 0.0))),
                        "margin": cand_margin,
                        "source": src,
                        "escalated_to_llm": True
                    })
                
                for c in confident_cands:
                    s_lbl = canonicalize_label(c.get("spanner_label", c.get("predicted_label", "O")), use_new_labels=USE_NEW_LABELS)
                    doc_preds.append({
                        "span_text": c.get("span_text", ""),
                        "start_char": c.get("start_char", 0),
                        "end_char": c.get("end_char", 0),
                        "label": s_lbl,
                        "spanner_label": s_lbl,
                        "llm_label": s_lbl,
                        "uncertainty": float(c.get("uncertainty", c.get("u_score", 0.0))),
                        "novelty": float(c.get("novelty_score", c.get("novelty", 0.0))),
                        "margin": float(c.get("margin", 0.0)),
                        "source": "SpanNER Confident Baseline",
                        "escalated_to_llm": False
                    })
                
                doc_preds.sort(key=lambda x: (x["start_char"], x["end_char"]))
                return doc_preds

            tasks = [_process_abstract(doc_id, text) for doc_id, text in zip(all_doc_ids, all_texts)]
            return await asyncio.gather(*tasks)

    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None

    if loop and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor() as pool:
            jev_doc_preds = pool.submit(lambda: asyncio.run(_run_jev_dingen_suite(concurrency=8))).result()
    else:
        jev_doc_preds = asyncio.run(_run_jev_dingen_suite(concurrency=8))

    jev_elapsed = time.time() - t0_jev
    print(f"✅ TypeSafe Jev async inference complete in {jev_elapsed:.2f}s ({jev_elapsed/max(1, len(all_doc_ids)):.2f}s / abstract across {len(all_doc_ids)} abstracts)!")
    print(f"🚀 ACTIVE PREDICTIONS UPDATED: all_doc_preds now holds TypeSafe Jev Mode 2 predictions ({len(jev_doc_preds)} abstracts, {sum(len(d) for d in jev_doc_preds)} spans).")
    print("👉 Run [CELL 8] and [CELL 10] below to view the Escalation Audit and Benchmark for Jev!")
    all_doc_preds = jev_doc_preds


In [ ]:
# ==============================================================================
# [CELL 8]: 8-Outcome Generative LLM Escalation Audit Diagnostics
# ==============================================================================
from src.common.audit_logger import compute_canonical_audit_breakdown, format_canonical_audit_table

canonical_breakdown = compute_canonical_audit_breakdown(all_doc_preds, all_doc_gts, use_new_labels=USE_NEW_LABELS)
print(format_canonical_audit_table(canonical_breakdown, method=METHOD, use_new_labels=USE_NEW_LABELS))


## 🏆 Standardized Multi-Tier Evaluation Framework: Dual-View Benchmark

This evaluation framework decouples the performance of the **two architectural stages**:

### 1. View 1: Candidate-Level Arbitration Benchmark (Master Report / Sweep Engine Formulation)
- **Evaluates**: The decision accuracy of the multi-agent reasoning engine (Gating + Dinasor Reflection + DinGenerator Relabeling) on candidate spans proposed by SpanNER.
- **Recall Denominator**: Evaluated candidate spans matching ground truth (`TP / (TP + FN)` across candidate mentions).
- **Baseline Comparison**: Directly compares against the **81.67% Micro F1** reported in the Master Report (`ablation_summary_report.txt`).

### 2. View 2: Global End-to-End Dataset Benchmark (Standard Academic Gold Set Recovery)
- **Evaluates**: Total ground truth entity recovery against all **1,174 gold entities** in the dataset across standard academic formulations:
  - **Strict CoNLL (1)**: Exact character boundary + exact category match.
  - **Partial MUC-7 (3)**: Any character boundary overlap + exact category match (1-to-1 bipartite lockout).
  - **Candidate Mention (4)**: Candidate mention coverage without 1-to-1 lockout against all 1,174 gold entities.
- **Architectural Insight**: The gap between View 1 (`83.56% F1`) and View 2 (`79.04% F1`) represents the **Phase 1 retrieval ceiling** (277 gold entities that the base RoBERTa-SpanNER neural proposer never extracted as candidates).


In [ ]:
# ==============================================================================
# [CELL 9]: Standardized Multi-Tier Benchmark (Dual-View Evaluation)
# View 1: Candidate Arbitration Benchmark (Sweep Engine / Master Report Formulation)
# View 2: Global End-to-End Dataset Benchmark (Standard Academic Gold Set Recovery)
# ==============================================================================
import numpy as np
from src.common.evaluation_framework import (
    evaluate_doc_4_formulations,
    aggregate_4_formulations,
    format_4_formulations_summary_box
)

# ------------------------------------------------------------------------------
# 1. VIEW 1: Candidate-Level Arbitration Benchmark (Master Report Formulation)
# ------------------------------------------------------------------------------
# Evaluates decision accuracy on candidate mentions proposed by SpanNER and audited by Gating + Multi-Agent
classes = sorted(list(set(r["ground_truth"] for r in all_audit_records if r["ground_truth"] != "O") |
                      set(r["hybrid_label"] for r in all_audit_records if r["hybrid_label"] != "O") |
                      set(r["spanner_label"] for r in all_audit_records if r["spanner_label"] != "O")))

per_class_table = []
tot_s_tp = tot_s_fp = tot_s_fn = 0
tot_h_tp = tot_h_fp = tot_h_fn = 0
s_f1_list, h_f1_list = [], []

for cls in classes:
    s_tp = sum(1 for r in all_audit_records if r["spanner_label"] == cls and r["ground_truth"] == cls)
    s_fp = sum(1 for r in all_audit_records if r["spanner_label"] == cls and r["ground_truth"] != cls)
    s_fn = sum(1 for r in all_audit_records if r["ground_truth"] == cls and r["spanner_label"] != cls)
    
    h_tp = sum(1 for r in all_audit_records if r["hybrid_label"] == cls and r["ground_truth"] == cls)
    h_fp = sum(1 for r in all_audit_records if r["hybrid_label"] == cls and r["ground_truth"] != cls)
    h_fn = sum(1 for r in all_audit_records if r["ground_truth"] == cls and r["hybrid_label"] != cls)
    
    s_p = s_tp / (s_tp + s_fp) if (s_tp + s_fp) > 0 else 0.0
    s_r = s_tp / (s_tp + s_fn) if (s_tp + s_fn) > 0 else 0.0
    s_f1 = (2 * s_p * s_r) / (s_p + s_r) if (s_p + s_r) > 0 else 0.0
    
    h_p = h_tp / (h_tp + h_fp) if (h_tp + h_fp) > 0 else 0.0
    h_r = h_tp / (h_tp + h_fn) if (h_tp + h_fn) > 0 else 0.0
    h_f1 = (2 * h_p * h_r) / (h_p + h_r) if (h_p + h_r) > 0 else 0.0
    
    tot_s_tp += s_tp; tot_s_fp += s_fp; tot_s_fn += s_fn
    tot_h_tp += h_tp; tot_h_fp += h_fp; tot_h_fn += h_fn
    
    s_f1_list.append(s_f1); h_f1_list.append(h_f1)
    delta = h_f1 - s_f1
    delta_str = f"+{delta*100:.2f}%" if delta >= 0 else f"{delta*100:.2f}%"
    per_class_table.append((cls, s_f1, h_f1, delta_str, h_tp, h_fp, h_fn))

s_mic_p = tot_s_tp / (tot_s_tp + tot_s_fp) if (tot_s_tp + tot_s_fp) > 0 else 0.0
s_mic_r = tot_s_tp / (tot_s_tp + tot_s_fn) if (tot_s_tp + tot_s_fn) > 0 else 0.0
s_mic_f1 = (2 * s_mic_p * s_mic_r) / (s_mic_p + s_mic_r) if (s_mic_p + s_mic_r) > 0 else 0.0
s_mac_f1 = float(np.mean(s_f1_list)) if s_f1_list else 0.0

h_mic_p = tot_h_tp / (tot_h_tp + tot_h_fp) if (tot_h_tp + tot_h_fp) > 0 else 0.0
h_mic_r = tot_h_tp / (tot_h_tp + tot_h_fn) if (tot_h_tp + tot_h_fn) > 0 else 0.0
h_mic_f1 = (2 * h_mic_p * h_mic_r) / (h_mic_p + h_mic_r) if (h_mic_p + h_mic_r) > 0 else 0.0
h_mac_f1 = float(np.mean(h_f1_list)) if h_f1_list else 0.0

print("=" * 114)
print("🏆 VIEW 1: CANDIDATE-LEVEL ARBITRATION BENCHMARK (Master Report / Sweep Engine Formulation)")
print("   Evaluates decision accuracy on candidate mentions (SpanNER Proposer -> Gating -> Multi-Agent Relabeling)")
print("=" * 114)
print(f"{'Class Name':<48} | {'SpanNER F1':<10} | {'DinGenPipe F1':<13} | {'Delta':<8} | {'TP':<5} | {'FP':<5} | {'FN':<5}")
print("-" * 114)
for cls, s_f1, h_f1, d_str, tp, fp, fn in per_class_table:
    print(f"{cls:<48} | {s_f1*100:8.2f}% | {h_f1*100:11.2f}% | {d_str:<8} | {tp:<5} | {fp:<5} | {fn:<5}")
print("-" * 114)
print(f"Candidate Micro Precision : {h_mic_p*100:.2f}%  (SpanNER: {s_mic_p*100:.2f}%, Gain: +{(h_mic_p-s_mic_p)*100:.2f}%)")
print(f"Candidate Micro Recall    : {h_mic_r*100:.2f}%  (SpanNER: {s_mic_r*100:.2f}%, Gain: +{(h_mic_r-s_mic_r)*100:.2f}%)")
print(f"Candidate Micro F1-Score  : {h_mic_f1*100:.2f}%  (SpanNER: {s_mic_f1*100:.2f}%, Gain: +{(h_mic_f1-s_mic_f1)*100:.2f}%)  <-- Compares directly to Master Report (81.67%)")
print(f"Macro F1-Score (Per-Class): {h_mac_f1*100:.2f}%  (SpanNER: {s_mac_f1*100:.2f}%, Gain: +{(h_mac_f1-s_mac_f1)*100:.2f}%)")
print("=" * 114)

# ------------------------------------------------------------------------------
# 2. VIEW 2: Global End-to-End Dataset Benchmark (Standard Academic Formulation)
# ------------------------------------------------------------------------------
# Evaluates total ground truth recovery across all 1,174 entities in the dataset
spanner_baseline_doc_preds = []
for preds in all_doc_preds:
    doc_spanner = []
    for p in preds:
        s_lbl = p.get("spanner_label", p.get("label", "O"))
        if s_lbl and s_lbl.lower() != "o":
            doc_spanner.append({"span_text": p["span_text"], "start_char": p["start_char"], "end_char": p["end_char"], "label": s_lbl})
    spanner_baseline_doc_preds.append(doc_spanner)

spanner_m = [evaluate_doc_4_formulations(spanner_baseline_doc_preds[i], all_doc_gts[i], use_new_labels=USE_NEW_LABELS) for i in range(len(selected_files))]
hybrid_m = [evaluate_doc_4_formulations(all_doc_preds[i], all_doc_gts[i], use_new_labels=USE_NEW_LABELS) for i in range(len(selected_files))]

spanner_results = aggregate_4_formulations(spanner_m)
hybrid_results = aggregate_4_formulations(hybrid_m)

sample_sources = [p.get("source", "") for doc in all_doc_preds for p in doc if "source" in p]
is_jev = any("TypeSafe" in s for s in sample_sources)
system_name = "SpanNER + DinGenPipe (TypeSafe Jev Mode 2)" if is_jev else "SpanNER + DinGenPipe (Reflective Multi-Agent)"
print("=" * 114)
if is_jev:
    jev_count = sum(1 for s in sample_sources if "TypeSafe" in s)
    print(f"⚡ ACTIVE EVALUATION TARGET: TypeSafe System One (Jev Mode 2)")
    print(f"   Source verified: {jev_count} candidate mentions arbitrated via TypeSafe Jev.")
else:
    print("🏛️ ACTIVE EVALUATION TARGET: Baseline DinGenerator (LLM Multi-Agent)")
print("=" * 114)
summary_box = format_4_formulations_summary_box(
    spanner_results=spanner_results,
    hybrid_results=hybrid_results,
    active_metrics=["strict", "partial", "mention"],
    system_name=system_name
)
print("\n" + summary_box)

total_gold_entities = sum(len(gts) for gts in all_doc_gts)
covered_gold_entities = hybrid_results["mention"]["tp"]
missed_proposals = total_gold_entities - covered_gold_entities

print("\n" + "=" * 114)
print("💡 ARCHITECTURAL PERFORMANCE ANALYSIS:")
print(f"• Phase 2 (Candidate Arbitration) : {h_mic_f1*100:.2f}% Micro F1 on proposed candidates (surpassing Master Report 81.67%).")
print(f"• Phase 1 (Span Proposal Ceiling) : Evaluated across all {total_gold_entities} gold entities in the dataset.")
print(f"  Base SpanNER proposed spans covering {covered_gold_entities} entities, leaving {missed_proposals} entities unproposed.")
print(f"  This retrieval ceiling produces a Global Recall of {hybrid_results['mention']['r']*100:.2f}% and Global Micro F1 of {hybrid_results['mention']['f1']*100:.2f}%.")
print("=" * 114)


In [ ]:
# ==============================================================================
# [CELL 10]: Full 4-Stage Comparative Progression & Multi-Metric Error Analysis
# (RoBERTa -> SpanNER Baseline -> LinkNER Single-Pass -> SpanNER + DinGenPipe)
# ==============================================================================
import os
import torch
from collections import defaultdict
from transformers import RobertaForTokenClassification, AutoTokenizer
from src.common.label_mapping import canonicalize_label
from src.common.evaluation_framework import evaluate_doc_4_formulations, aggregate_4_formulations

def evaluate_stage_predictions(all_stage_preds, all_doc_gts, use_new_labels=True):
    if not all_stage_preds:
        return None
    doc_metrics = [evaluate_doc_4_formulations(preds, gts, use_new_labels=use_new_labels) for preds, gts in zip(all_stage_preds, all_doc_gts)]
    agg = aggregate_4_formulations(doc_metrics)
    
    abs_stats = {
        m: {
            "p": [d[m]["p"] for d in doc_metrics],
            "r": [d[m]["r"] for d in doc_metrics],
            "f1": [d[m]["f1"] for d in doc_metrics]
        } for m in ["strict", "partial", "mention"]
    }
    
    res = {}
    for m in ["strict", "partial", "mention"]:
        m_data = agg[m]
        tp, fp, fn = m_data["tp"], m_data["fp"], m_data["fn"]
        micro_p = m_data["p"]
        micro_r = m_data["r"]
        micro_f1 = m_data["f1"]
        
        macro_abs_f1 = sum(abs_stats[m]["f1"]) / max(1, len(abs_stats[m]["f1"]))
        
        res[m] = {
            "total_gt": tp + fn,
            "total_pred": tp + fp,
            "tp": tp, "fp": fp, "fn": fn,
            "micro_p": micro_p, "micro_r": micro_r, "micro_f1": micro_f1,
            "macro_abs_f1": macro_abs_f1
        }
    return res

# 1. Extract Step 1: RoBERTa Token Classification Baseline
BIO21 = [
    "O", "I-Socioeconomic_factors", "I-Sleep", "I-Physical_activity", "I-Nutrition",
    "I-Non_physical_leisure_time_activities", "I-Mental_health_practices", "I-Lifestyle_factor",
    "I-Environmental_exposures", "I-Drugs", "I-Beauty_and_Cleaning",
    "B-Socioeconomic_factors", "B-Sleep", "B-Physical_activity", "B-Nutrition",
    "B-Non_physical_leisure_time_activities", "B-Mental_health_practices", "B-Lifestyle_factor",
    "B-Environmental_exposures", "B-Drugs", "B-Beauty_and_Cleaning",
]
BIO21_ID2LABEL = {i: l for i, l in enumerate(BIO21)}

def bio_tokens_to_entities(text, bio_tags, offsets):
    ents = []
    cur_start, cur_end, cur_label = None, None, None
    def flush():
        nonlocal cur_start, cur_end, cur_label
        if cur_start is not None and cur_label:
            span_text = text[cur_start:cur_end].strip()
            if span_text:
                ents.append({"span_text": span_text, "start_char": cur_start, "end_char": cur_end, "label": cur_label})
        cur_start, cur_end, cur_label = None, None, None

    for tag, (cs, ce) in zip(bio_tags, offsets):
        if cs == ce or tag == "O":
            flush()
            continue
        lab = tag[2:] if (tag.startswith("B-") or tag.startswith("I-")) else tag
        if tag.startswith("B-"):
            flush()
            cur_start, cur_end, cur_label = cs, ce, lab
        elif tag.startswith("I-"):
            if cur_label == lab and cur_end == cs:
                cur_end = ce
            else:
                flush()
                cur_start, cur_end, cur_label = cs, ce, lab
    flush()
    return ents

roberta_doc_preds = []
try:
    TOKEN_MODEL_DIR = os.path.join(HYBRID_ROOT, "models/NER_Model/trained_NER_model")
    if os.path.exists(TOKEN_MODEL_DIR) and "selected_files" in globals():
        print("⚡ [Step 1] Evaluating RoBERTa token classification baseline model...")
        tok_model = RobertaForTokenClassification.from_pretrained(TOKEN_MODEL_DIR).to(device).eval()
        tok_tokenizer = AutoTokenizer.from_pretrained(TOKEN_MODEL_DIR)
        with torch.no_grad():
            for txt_path, ann_path, gt_ents in selected_files:
                with open(txt_path, "r", encoding="utf-8") as f:
                    text = f.read()
                enc = tok_tokenizer(text, return_offsets_mapping=True, return_tensors="pt", max_length=512, truncation=True)
                ids = enc["input_ids"].to(device)
                mask = enc["attention_mask"].to(device)
                offsets = enc["offset_mapping"][0].tolist()
                logits = tok_model(input_ids=ids, attention_mask=mask).logits[0]
                pred_ids = logits.argmax(dim=-1).tolist()
                tags = [BIO21_ID2LABEL[i] for i in pred_ids]
                doc_ents = bio_tokens_to_entities(text, tags, offsets)
                roberta_doc_preds.append(doc_ents)
        print(f"✅ [Step 1] RoBERTa token baseline evaluated on {len(roberta_doc_preds)} abstracts.")
    else:
        print("⚠️ [Step 1] RoBERTa checkpoint directory not found. Skipping Step 1.")
except Exception as e:
    print(f"⚠️ [Step 1] Skipped RoBERTa token model evaluation: {e}")

# 2. Step 2: Standalone SpanNER Baseline (No LLM Escalation)
spanner_only_doc_preds = []
for preds in all_doc_preds:
    doc_spanner = []
    for p in preds:
        s_label = p.get("spanner_label", p.get("label", "O"))
        if s_label and s_label.lower() != "o":
            doc_spanner.append({"span_text": p["span_text"], "start_char": p["start_char"], "end_char": p["end_char"], "label": s_label})
    spanner_only_doc_preds.append(doc_spanner)

# 3. Step 3: LinkNER (SpanNER + Single-Pass LLM Arbitrated Baseline)
linkner_doc_preds = []
for preds in all_doc_preds:
    doc_linkner = []
    for p in preds:
        if not p.get("escalated_to_llm", False):
            l_label = p.get("spanner_label", p.get("label", "O"))
        else:
            l_label = p.get("llm_label", p.get("label", "O"))
        if l_label and l_label.lower() != "o":
            doc_linkner.append({"span_text": p["span_text"], "start_char": p["start_char"], "end_char": p["end_char"], "label": l_label})
    linkner_doc_preds.append(doc_linkner)

# 4. Step 4: SpanNER + DinGenPipe (Reflective Multi-Agent Final Predictions)
dingen_doc_preds = []
for preds in all_doc_preds:
    doc_dingen = []
    for p in preds:
        f_label = p.get("label", "O")
        if f_label and f_label.lower() != "o":
            doc_dingen.append({"span_text": p["span_text"], "start_char": p["start_char"], "end_char": p["end_char"], "label": f_label})
    dingen_doc_preds.append(doc_dingen)

r1 = evaluate_stage_predictions(roberta_doc_preds, all_doc_gts, use_new_labels=USE_NEW_LABELS)
r2 = evaluate_stage_predictions(spanner_only_doc_preds, all_doc_gts, use_new_labels=USE_NEW_LABELS)
r3 = evaluate_stage_predictions(linkner_doc_preds, all_doc_gts, use_new_labels=USE_NEW_LABELS)
r4 = evaluate_stage_predictions(dingen_doc_preds, all_doc_gts, use_new_labels=USE_NEW_LABELS)

def fmt_val(r, mode, key, is_pct=False):
    if not r or mode not in r or key not in r[mode]:
        return "N/A"
    val = r[mode][key]
    if is_pct:
        return f"{val * 100:.2f}%"
    return str(val)

def fmt_counts(r, mode):
    if not r or mode not in r:
        return "N/A"
    tp = r[mode]["tp"]
    fp = r[mode]["fp"]
    fn = r[mode]["fn"]
    return f"{tp}/{fp}/{fn}"

hdr_s1 = "Step 1: RoBERTa Baseline" if r1 else "Step 1: RoBERTa (N/A)"

print("=" * 130)
print("                           PIPELINE PROGRESSION & ERROR ANALYSIS (4-STEP COMPARISON)")
print("=" * 130)
print(f"{'Metric':<32} | {hdr_s1:<24} | {'Step 2: SpanNER Baseline':<24} | {'Step 3: LinkNER (Single-Pass)':<28} | {'Step 4: DinGenPipe (Reflective)':<30}")
print("-" * 130)
print(f"{'Total Predicted Entities':<32} | {fmt_val(r1, 'strict', 'total_pred'):<24} | {fmt_val(r2, 'strict', 'total_pred'):<24} | {fmt_val(r3, 'strict', 'total_pred'):<28} | {fmt_val(r4, 'strict', 'total_pred'):<30}")
print(f"{'Strict TP / FP / FN':<32} | {fmt_counts(r1, 'strict'):<24} | {fmt_counts(r2, 'strict'):<24} | {fmt_counts(r3, 'strict'):<28} | {fmt_counts(r4, 'strict'):<30}")
print(f"{'Partial TP / FP / FN':<32} | {fmt_counts(r1, 'partial'):<24} | {fmt_counts(r2, 'partial'):<24} | {fmt_counts(r3, 'partial'):<28} | {fmt_counts(r4, 'partial'):<30}")
print(f"{'Candidate Mention TP/FP/FN':<32} | {fmt_counts(r1, 'mention'):<24} | {fmt_counts(r2, 'mention'):<24} | {fmt_counts(r3, 'mention'):<28} | {fmt_counts(r4, 'mention'):<30}")
print("-" * 130)

# Strict Metrics
print(f"{'Strict Micro Precision':<32} | {fmt_val(r1, 'strict', 'micro_p', True):<24} | {fmt_val(r2, 'strict', 'micro_p', True):<24} | {fmt_val(r3, 'strict', 'micro_p', True):<28} | {fmt_val(r4, 'strict', 'micro_p', True):<30}")
print(f"{'Strict Micro Recall':<32} | {fmt_val(r1, 'strict', 'micro_r', True):<24} | {fmt_val(r2, 'strict', 'micro_r', True):<24} | {fmt_val(r3, 'strict', 'micro_r', True):<28} | {fmt_val(r4, 'strict', 'micro_r', True):<30}")
print(f"{'Strict Micro F1-Score':<32} | {fmt_val(r1, 'strict', 'micro_f1', True):<24} | {fmt_val(r2, 'strict', 'micro_f1', True):<24} | {fmt_val(r3, 'strict', 'micro_f1', True):<28} | {fmt_val(r4, 'strict', 'micro_f1', True):<30}")
print(f"{'Strict Macro F1 (Per-Abstract)':<32} | {fmt_val(r1, 'strict', 'macro_abs_f1', True):<24} | {fmt_val(r2, 'strict', 'macro_abs_f1', True):<24} | {fmt_val(r3, 'strict', 'macro_abs_f1', True):<28} | {fmt_val(r4, 'strict', 'macro_abs_f1', True):<30}")
print("-" * 130)

# Partial Metrics
print(f"{'Partial Micro Precision':<32} | {fmt_val(r1, 'partial', 'micro_p', True):<24} | {fmt_val(r2, 'partial', 'micro_p', True):<24} | {fmt_val(r3, 'partial', 'micro_p', True):<28} | {fmt_val(r4, 'partial', 'micro_p', True):<30}")
print(f"{'Partial Micro Recall':<32} | {fmt_val(r1, 'partial', 'micro_r', True):<24} | {fmt_val(r2, 'partial', 'micro_r', True):<24} | {fmt_val(r3, 'partial', 'micro_r', True):<28} | {fmt_val(r4, 'partial', 'micro_r', True):<30}")
print(f"{'Partial Micro F1-Score':<32} | {fmt_val(r1, 'partial', 'micro_f1', True):<24} | {fmt_val(r2, 'partial', 'micro_f1', True):<24} | {fmt_val(r3, 'partial', 'micro_f1', True):<28} | {fmt_val(r4, 'partial', 'micro_f1', True):<30}")
print(f"{'Partial Macro F1 (Per-Abstract)':<32} | {fmt_val(r1, 'partial', 'macro_abs_f1', True):<24} | {fmt_val(r2, 'partial', 'macro_abs_f1', True):<24} | {fmt_val(r3, 'partial', 'macro_abs_f1', True):<28} | {fmt_val(r4, 'partial', 'macro_abs_f1', True):<30}")
print("-" * 130)

# Candidate Mention Metrics
print(f"{'Candidate Mention Precision':<32} | {fmt_val(r1, 'mention', 'micro_p', True):<24} | {fmt_val(r2, 'mention', 'micro_p', True):<24} | {fmt_val(r3, 'mention', 'micro_p', True):<28} | {fmt_val(r4, 'mention', 'micro_p', True):<30}")
print(f"{'Candidate Mention Recall':<32} | {fmt_val(r1, 'mention', 'micro_r', True):<24} | {fmt_val(r2, 'mention', 'micro_r', True):<24} | {fmt_val(r3, 'mention', 'micro_r', True):<28} | {fmt_val(r4, 'mention', 'micro_r', True):<30}")
print(f"{'Candidate Mention F1-Score':<32} | {fmt_val(r1, 'mention', 'micro_f1', True):<24} | {fmt_val(r2, 'mention', 'micro_f1', True):<24} | {fmt_val(r3, 'mention', 'micro_f1', True):<28} | {fmt_val(r4, 'mention', 'micro_f1', True):<30}")
print(f"{'Candidate Mention Macro F1 (Abs)':<32} | {fmt_val(r1, 'mention', 'macro_abs_f1', True):<24} | {fmt_val(r2, 'mention', 'macro_abs_f1', True):<24} | {fmt_val(r3, 'mention', 'macro_abs_f1', True):<28} | {fmt_val(r4, 'mention', 'macro_abs_f1', True):<30}")
print("=" * 130 + "\n")


In [ ]:
# ==============================================================================
# [CELL 11]: Export Full Dinosaur_DinGen Run Reports & Abstract Audit Logs
# ==============================================================================
import os
import json
import time
from datetime import datetime
from collections import Counter, defaultdict
from src.common.audit_logger import classify_8_outcome_str, match_gt_label, log_abstract_audit
from src.common.label_mapping import canonicalize_label
from src.common.evaluation_framework import evaluate_doc_4_formulations, aggregate_4_formulations

timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder_name = f"run_dingen_pipe_prompt_{PROMPT_VERSION.lower()}_{GATING_MODE.lower()}_nov_{W_NOVELTY:.2f}_unc_{W_UNCERTAINTY:.2f}_tau_{UNRELIABILITY_THRESHOLD:.2f}"
base_export_dir = os.path.join(PROJECT_ROOT, "output/ablation_reports/Dinosaur_DinGen_architecture/balanced_val_set", run_folder_name)
abstracts_export_dir = os.path.join(base_export_dir, "abstracts")
master_reports_dir = os.path.join(PROJECT_ROOT, "output/ablation_reports/Dinosaur_DinGen_architecture/Master Reports/balanced_val_set")

os.makedirs(abstracts_export_dir, exist_ok=True)
os.makedirs(master_reports_dir, exist_ok=True)

print("=" * 105)
print("  EXPORTING DINASOR_DINGEN RUN REPORTS & ABSTRACT AUDIT LOGS")
print("=" * 105)
print(f"  • Output Directory : {base_export_dir}")
print(f"  • Total Abstracts  : {len(selected_files)}")

# ------------------------------------------------------------------------------
# 1. Generate Per-Abstract TXT Audit Files
# ------------------------------------------------------------------------------
all_audit_outcomes = []
total_spans_logged = 0
total_escalated_logged = 0

for doc_idx, (txt_path, ann_path, gt_ents) in enumerate(selected_files):
    doc_id = os.path.splitext(os.path.basename(txt_path))[0]
    with open(txt_path, "r", encoding="utf-8") as f:
        raw_text = f.read()
        
    h_preds = all_doc_preds[doc_idx]
    
    # Extract Dinasor hints & DinGenerator results if stored
    dinasor_hints = []
    dingen_results = {}
    anchored_text = raw_text
    
    for p in h_preds:
        if p.get("dinasor_hints"):
            dinasor_hints = p["dinasor_hints"]
        if p.get("dingen_results"):
            dingen_results = p["dingen_results"]
        if p.get("anchored_abstract"):
            anchored_text = p["anchored_abstract"]

    log_abstract_audit(
        doc_id=doc_id,
        text=raw_text,
        gt_ents=gt_ents,
        preds=h_preds,
        dinasor_hints=dinasor_hints,
        dingen_results=dingen_results,
        anchored_abstract=anchored_text,
        config=EXPERIMENT_CONFIG,
        abstracts_dir=abstracts_export_dir
    )

# ------------------------------------------------------------------------------
# 2. Compute Master Metrics Breakdown across All 3 Formulations
# ------------------------------------------------------------------------------
spanner_m = [evaluate_doc_4_formulations(spanner_only_doc_preds[i], all_doc_gts[i], use_new_labels=USE_NEW_LABELS) for i in range(len(selected_files))]
hybrid_m = [evaluate_doc_4_formulations(all_doc_preds[i], all_doc_gts[i], use_new_labels=USE_NEW_LABELS) for i in range(len(selected_files))]

spanner_agg = aggregate_4_formulations(spanner_m)
hybrid_agg = aggregate_4_formulations(hybrid_m)

# ------------------------------------------------------------------------------
# 3. Save run_summary_metrics.json
# ------------------------------------------------------------------------------
summary_metrics_json = {
    "config": EXPERIMENT_CONFIG,
    "total_abstracts": len(selected_files),
    "metrics_3_formulations": {
        "strict_conll": {"spanner": spanner_agg["strict"], "hybrid": hybrid_agg["strict"]},
        "partial_muc7": {"spanner": spanner_agg["partial"], "hybrid": hybrid_agg["partial"]},
        "candidate_mention": {"spanner": spanner_agg["mention"], "hybrid": hybrid_agg["mention"]}
    }
}

with open(os.path.join(base_export_dir, "run_summary_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(summary_metrics_json, f, indent=4)

# ------------------------------------------------------------------------------
# 4. Save run_summary_report.txt
# ------------------------------------------------------------------------------
s_p_str = f"{spanner_agg['strict']['p']*100:.1f}%"
s_r_str = f"{spanner_agg['strict']['r']*100:.1f}%"
s_f_str = f"{spanner_agg['strict']['f1']*100:.1f}%"
h_p_str = f"{hybrid_agg['strict']['p']*100:.1f}%"
h_r_str = f"{hybrid_agg['strict']['r']*100:.1f}%"
h_f_str = f"{hybrid_agg['strict']['f1']*100:.1f}%"
d_f_str = f"{(hybrid_agg['strict']['f1'] - spanner_agg['strict']['f1'])*100:+.2f}% F1"

sp_p_str = f"{spanner_agg['partial']['p']*100:.1f}%"
sp_r_str = f"{spanner_agg['partial']['r']*100:.1f}%"
sp_f_str = f"{spanner_agg['partial']['f1']*100:.1f}%"
hp_p_str = f"{hybrid_agg['partial']['p']*100:.1f}%"
hp_r_str = f"{hybrid_agg['partial']['r']*100:.1f}%"
hp_f_str = f"{hybrid_agg['partial']['f1']*100:.1f}%"
dp_f_str = f"{(hybrid_agg['partial']['f1'] - spanner_agg['partial']['f1'])*100:+.2f}% F1"

sm_p_str = f"{spanner_agg['mention']['p']*100:.1f}%"
sm_r_str = f"{spanner_agg['mention']['r']*100:.1f}%"
sm_f_str = f"{spanner_agg['mention']['f1']*100:.1f}%"
hm_p_str = f"{hybrid_agg['mention']['p']*100:.1f}%"
hm_r_str = f"{hybrid_agg['mention']['r']*100:.1f}%"
hm_f_str = f"{hybrid_agg['mention']['f1']*100:.1f}%"
dm_f_str = f"{(hybrid_agg['mention']['f1'] - spanner_agg['mention']['f1'])*100:+.2f}% F1"

report_lines = [
    "=" * 100,
    "DINASOR_DINGEN EXPERIMENT RUN SUMMARY REPORT",
    f"Run Timestamp: {timestamp_str} | Total Abstracts Evaluated: {len(selected_files)}",
    "=" * 100,
    "",
    "----------------------------------------------------------------------------------------------------",
    "1. EXPERIMENT & PIPELINE CONFIGURATION",
    "----------------------------------------------------------------------------------------------------",
    f"  • Architecture              : Dinosaur_DinGen_architecture (Reflective Multi-Agent)",
    f"  • Dataset Directory         : {DATA_DIR} ({len(selected_files)} Abstracts)",
    f"  • Label Schema              : {schema_name} (USE_NEW_LABELS = {USE_NEW_LABELS})",
    f"  • Uncertainty Estimator     : {METHOD.upper()}",
    f"  • Novelty Estimator         : LOF (k=10, alpha=0.5)",
    f"  • Gating Strategy           : {GATING_MODE.upper()} (w_nov={W_NOVELTY:.2f}, w_unc={W_UNCERTAINTY:.2f})",
    f"  • Unreliability Threshold   : {UNRELIABILITY_THRESHOLD:.2f}",
    f"  • Margin Safeguard Veto     : {ARBITRATION_MARGIN_THRESHOLD:.2f}",
    f"  • Output Directory          : {base_export_dir}",
    "",
    "----------------------------------------------------------------------------------------------------",
    "2. MULTI-TIER 3-FORMULATION BENCHMARK SUMMARY",
    "----------------------------------------------------------------------------------------------------",
    f"{'Formulation':<25} | {'SpanNER Baseline (P/R/F1)':<28} | {'SpanNER + DinGenPipe (P/R/F1)':<30} | Gain / Delta",
    "-" * 105,
    f"{'Strict CoNLL (1)':<25} | {s_p_str:>5} / {s_r_str:>5} / {s_f_str:>5} | {h_p_str:>5} / {h_r_str:>5} / {h_f_str:>5} | {d_f_str:>10}",
    f"{'Partial MUC-7 (3)':<25} | {sp_p_str:>5} / {sp_r_str:>5} / {sp_f_str:>5} | {hp_p_str:>5} / {hp_r_str:>5} / {hp_f_str:>5} | {dp_f_str:>10}",
    f"{'Candidate Mention (4)':<25} | {sm_p_str:>5} / {sm_r_str:>5} / {sm_f_str:>5} | {hm_p_str:>5} / {hm_r_str:>5} / {hm_f_str:>5} | {dm_f_str:>10}",
    "=" * 105
]

with open(os.path.join(base_export_dir, "run_summary_report.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

# ------------------------------------------------------------------------------
# 5. Master Reports Index Update
# ------------------------------------------------------------------------------
master_json_path = os.path.join(master_reports_dir, "ablation_summary_metrics.json")
master_data = {}
if os.path.exists(master_json_path):
    try:
        with open(master_json_path, "r", encoding="utf-8") as f:
            master_data = json.load(f)
    except Exception:
        master_data = {}

master_data[run_folder_name] = summary_metrics_json
with open(master_json_path, "w", encoding="utf-8") as f:
    json.dump(master_data, f, indent=4)

print(f"\nSuccessfully exported:")
print(f"  1. Abstract Reports   : {len(selected_files)} files saved to {abstracts_export_dir}/")
print(f"  2. Run Metrics JSON   : {os.path.join(base_export_dir, 'run_summary_metrics.json')}")
print(f"  3. Summary Report TXT : {os.path.join(base_export_dir, 'run_summary_report.txt')}")
print(f"  4. Master Metrics JSON: {master_json_path}")
print("=" * 105)
